# PK/PD Take-Home Exercises

> **Before running anything, read this section.**
>
> This notebook uses R. To run it you need:
>
> 1. **R installed**: Download from https://cran.r-project.org (choose your OS). Version 4.2 or later recommended.
> 2. **R kernel for Jupyter**: After installing R, open an R console and run:
>    ```r
>    install.packages("IRkernel")
>    IRkernel::installspec()
>    ```
>    Then restart Jupyter. You should see "R" as a kernel option when creating a new notebook.
> 3. **Packages**: Run the setup cell below. It will try to restore packages using `renv` (which ensures you get the exact same package versions as the lecture). If that fails, it falls back to installing directly.
>
> If you hit any issues, let me know.

In [1]:
# ── Package setup ────────────────────────────────────────────────────────────
# We use renv to ensure everyone has the same package versions.
# If renv fails for any reason, we fall back to direct installation.

setup_ok <- tryCatch({
  # Install renv itself if not present
  if (!requireNamespace("renv", quietly = TRUE)) {
    install.packages("renv", repos = "https://cloud.r-project.org")
  }
  # Restore packages from the lockfile in the project root
  renv::restore(prompt = FALSE)
  TRUE
}, error = function(e) {
  message("renv restore failed: ", conditionMessage(e))
  message("Falling back to install.packages()...")
  FALSE
})

if (!setup_ok) {
  pkgs_needed <- c("ggplot2", "dplyr", "tidyr", "nlme", "deSolve", "gridExtra")
  missing     <- pkgs_needed[!sapply(pkgs_needed, requireNamespace, quietly = TRUE)]
  if (length(missing) > 0) {
    install.packages(missing, repos = "https://cloud.r-project.org")
  }
}

# Load packages (same for both paths)
library(ggplot2)
library(dplyr)
library(deSolve)
library(nlme)
library(gridExtra)

options(repr.plot.width = 9, repr.plot.height = 4)
cat("Setup complete.\n")

- The library is already synchronized with the lockfile.



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:

    collapse



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




Setup complete.


In [2]:
# ── Helper functions (from the lecture — do not modify) ─────────────────────

# 1-compartment IV bolus: analytic solution
pk_iv_bolus <- function(t, dose, CL, V) {
  ke <- CL / V
  (dose / V) * exp(-ke * t)
}

# 1-compartment oral absorption: analytic solution
# F = bioavailability (default 1)
pk_oral <- function(t, dose, ka, CL, V, F = 1) {
  ke <- CL / V
  if (abs(ka - ke) < 1e-6) ka <- ka * 1.0001
  (F * dose * ka) / (V * (ka - ke)) * (exp(-ke * t) - exp(-ka * t))
}

# Model formula for nls() fitting
one_comp_oral <- function(Time, Dose, ka, CL, V) {
  ke <- CL / V
  (Dose * ka) / (V * (ka - ke)) * (exp(-ke * Time) - exp(-ka * Time))
}

# Multiple oral dosing by superposition
pk_multidose <- function(t_vec, dose, ka, CL, V, interval, n_doses) {
  sapply(t_vec, function(t) {
    dose_times <- seq(0, (n_doses - 1) * interval, by = interval)
    total <- 0
    for (t0 in dose_times) {
      if (t >= t0) total <- total + pk_oral(t - t0, dose, ka, CL, V)
    }
    total
  })
}

# Emax PD model
emax_model <- function(C, Emax, EC50) {
  Emax * C / (EC50 + C)
}

cat("All helper functions loaded.\n")

All helper functions loaded.


In [3]:
# Load the built-in Theoph dataset
data(Theoph)

# Quick look
cat("Dimensions:", nrow(Theoph), "rows x", ncol(Theoph), "columns\n")
cat("Subjects:", length(unique(Theoph$Subject)), "\n")
head(Theoph)

Dimensions: 132 rows x 5 columns
Subjects: 12 


,Subject,Wt,Dose,Time,conc
,<ord>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,79.6,4.02,0.00,0.74
2,1,79.6,4.02,0.25,2.84
3,1,79.6,4.02,0.57,6.57
4,1,79.6,4.02,1.12,10.50
5,1,79.6,4.02,2.02,9.66
6,1,79.6,4.02,3.82,8.58


**For reference**, the lecture fitted Subject 1 and found:

| Parameter | Estimate | Meaning |
|---|---|---|
| ka | 1.777 h⁻¹ | Absorption rate constant |
| CL | 0.0199 L/kg/h | Clearance |
| V | 0.369 L/kg | Volume of distribution |
| ke = CL/V | 0.054 h⁻¹ | Elimination rate constant |
| t½ | 12.8 h | Half-life |
| Tmax | ~2.0 h | Predicted time of peak |

You will use these for comparison in several exercises.

## Exercise 1: PK Parameters — Intuition and Interpretation

In the lecture we saw that half-life depends on both clearance (CL) and volume of distribution (V). In this exercise you will explore that relationship by simulating IV bolus profiles for three hypothetical drugs and reasoning about what the numbers mean.

**Scenario**: You are evaluating three candidate drugs for a once-daily oral regimen. All three are given as a 100 mg IV bolus. Their PK parameters are:

| Drug | CL (L/h) | V (L) |
|---|---|---|
| Drug A | 10 | 20 |
| Drug B | 10 | 100 |
| Drug C | 2 | 20 |

**Note**: these are *total* parameters (not per-kg), just for this exercise.

In [4]:
# Exercise 1a: Simulate IV bolus profiles for Drugs A, B, and C
# The function pk_iv_bolus(t, dose, CL, V) is already defined above.

t_seq <- seq(0, 48, by = 0.1)  # 48-hour time window
dose  <- 100  # mg

# Build a data frame with all three drugs
# Hint: use expand.grid() to create all combinations of t and drug,
# then add the correct CL and V for each drug.

drugs <- data.frame(
  Drug = c("Drug A", "Drug B", "Drug C"),
  CL   = c(10, 10, ___),   # fill in Drug C's CL
  V    = c(___, 100, 20)   # fill in Drug A's V
)

# Compute the concentration at each time for each drug
sim_df <- merge(data.frame(t = t_seq), drugs) %>%
  mutate(
    C      = pk_iv_bolus(t, dose, CL, V),
    ke     = CL / V,
    t_half = round(log(2) / ke, 1)
  )

# Plot
ggplot(sim_df, aes(x = t, y = C, color = Drug)) +
  geom_line(linewidth = 1) +
  labs(
    title = "IV Bolus Profiles: Three Candidate Drugs",
    x = "Time (h)", y = "Concentration (mg/L)", color = NULL
  ) +
  theme_minimal()

ERROR: Error in parse(text = input): <text>:13:21: unexpected input
12:   Drug = c("Drug A", "Drug B", "Drug C"),
13:   CL   = c(10, 10, __
                        ^


In [ ]:
# Exercise 1b: Print the half-life for each drug
# Fill in the formula: t_half = log(2) / ke, where ke = CL / V

drugs <- drugs %>%
  mutate(
    ke     = CL / V,
    t_half = ___         # YOUR CODE HERE: formula for half-life
  )

print(drugs)

**Question 1a**: Rank the three drugs from longest to shortest half-life. Which parameter (CL or V) is responsible for the difference between Drug A and Drug B? Which is responsible for the difference between Drug A and Drug C?

*Your answer:*
[Write here]

---

**Question 1b**: Drug B and Drug C have very different CL and V values, but do their half-lives end up similar or different? What does this tell you about using half-life alone to describe PK?

*Your answer:*
[Write here]

---

**Question 1c**: You need to choose a drug for a once-daily regimen. The therapeutic window is 1–10 mg/L. Based on the plot, which drug(s) would still have concentrations above 1 mg/L at 24 hours? Which would you choose, and why?

*Your answer:*
[Write here]

## Exercise 2: Fitting the Oral PK Model to a New Subject

In the lecture we fitted the one-compartment oral model to Subject 1. Now you will repeat that analysis for **Subject 5** — a subject who, as you will see, has notably different PK from Subject 1.

**Your task**:
1. Extract Subject 5's data and plot their concentration–time profile
2. Fit the oral PK model using `nls()`
3. Plot observed vs predicted
4. Compare the fitted parameters to Subject 1

In [ ]:
# Extract Subject 5 data
sub5 <- subset(Theoph, Subject == ___)    # fill in subject number

# Plot Subject 5's profile
ggplot(sub5, aes(x = Time, y = conc)) +
  geom_point(size = 3, color = "steelblue") +
  geom_line(alpha = 0.4) +
  labs(
    title = "Subject ___: Concentration-Time Profile",   # fill in subject number
    x = "Time (h)", y = "Concentration (mg/L)"
  ) +
  theme_minimal()

In [ ]:
# Fit the one-compartment oral model to Subject 5
# Starting values: use the same ones from the lecture as a starting point
# (they should be close enough for nls to converge)

fit_sub5 <- nls(
  conc ~ one_comp_oral(Time, Dose, ka, CL, V),
  data  = ___,                              # which data frame?
  start = c(ka = 1.5, CL = ___, V = ___)   # use lecture starting values
)

summary(fit_sub5)

In [ ]:
# Extract parameters and compute derived quantities
params5 <- coef(fit_sub5)

ke5     <- params5["CL"] / params5["V"]
t_half5 <- log(2) / ke5
tmax5   <- log(params5["ka"] / ke5) / (params5["ka"] - ke5)

cat("=== Fitted PK Parameters: Subject 5 vs Subject 1 ===\n")
cat(sprintf("          Subject 1   Subject 5\n"))
cat(sprintf("  ka      1.777       %.3f  h-1\n",   params5["ka"]))
cat(sprintf("  CL      0.0199      %.4f  L/kg/h\n", params5["CL"]))
cat(sprintf("  V       0.369       %.3f  L/kg\n",   params5["V"]))
cat(sprintf("  t1/2    12.8        %.1f   h\n",     t_half5))
cat(sprintf("  Tmax    2.0         %.1f   h\n",     tmax5))

In [ ]:
# Plot observed vs predicted for Subject 5
t_fine <- seq(0, 25, by = 0.1)

C_pred5 <- pk_oral(t_fine,
                   dose = sub5$Dose[1],
                   ka   = params5[["ka"]],
                   CL   = params5[["CL"]],
                   V    = params5[["V"]])

pred_df5 <- data.frame(Time = t_fine, conc = C_pred5)

ggplot() +
  geom_point(data = sub5,    aes(x = Time, y = conc), size = 3, color = "steelblue") +
  geom_line( data = pred_df5, aes(x = Time, y = conc), color = "firebrick", linewidth = 1) +
  labs(
    title = "Subject 5: Observed vs Fitted Model",
    x = "Time (h)", y = "Concentration (mg/L)"
  ) +
  theme_minimal()

**Question 2a**: Compare the CL values for Subject 1 and Subject 5. Which subject clears theophylline faster? By how much (express as a ratio or percentage)?

*Your answer:*
[Write here]

---

**Question 2b**: Theophylline is primarily metabolized by the liver enzyme CYP1A2. Cigarette smoking is a potent inducer of CYP1A2 (it increases the amount of enzyme). Based on the CL difference you found, which subject do you think is more likely to be a smoker? What would you want to check to confirm this hypothesis?

*Your answer:*
[Write here]

---

**Question 2c**: Look at the Tmax values for both subjects. What does a shorter Tmax tell you about the absorption process? Are the ka values consistent with this?

*Your answer:*
[Write here]

---

**Question 2d** *(stretch)*: Subject 5 has a higher dose than Subject 1 (check `sub5$Dose[1]`). Does a higher dose affect the fitted CL or V? Why or why not?

*Your answer:*
[Write here]

## Exercise 3: Designing a Dosing Regimen for Subject 5

Now that you have Subject 5's PK parameters, you can simulate what happens when the drug is given repeatedly. The **therapeutic window for theophylline** is approximately:

- **Minimum effective concentration (MEC)**: 5 mg/L
- **Maximum tolerated concentration (MTC)**: 15 mg/L (toxicity risk above this)

Note: these are plasma concentrations, not dose-normalized. The dose in the dataset is in mg/kg, and concentrations are in mg/L. When simulating, use the *actual dose in mg* (dose × body weight). Subject 5's weight is in `sub5$Wt[1]`.

**Your task**: Simulate three dosing regimens (q8h, q12h, q24h) for Subject 5 over 72 hours at the same per-kg dose as in the study. Determine which regimen keeps concentrations within the 5–15 mg/L therapeutic window at steady state.

In [ ]:
# Subject 5's dose in mg (per-kg dose × body weight)
dose_mg <- sub5$Dose[1] * ___    # YOUR CODE: multiply by body weight

cat(sprintf("Subject 5 actual dose: %.1f mg\n", dose_mg))
cat(sprintf("Body weight: %.1f kg\n", sub5$Wt[1]))

t_md <- seq(0, 72, by = 0.1)

# Simulate three dosing intervals
# The pk_multidose function takes: t_vec, dose, ka, CL, V, interval, n_doses
# For CL and V: note these are in L/kg — multiply by body weight to get L
# (The pk_multidose function uses total dose in mg, so CL and V should be in L/h and L)

CL_total <- params5[["CL"]] * sub5$Wt[1]   # L/h (total, not per kg)
V_total  <- params5[["V"]]  * ___           # YOUR CODE: total V in L

intervals <- c(8, 12, 24)

sim_list <- lapply(intervals, function(tau) {
  data.frame(
    Time     = t_md,
    conc     = pk_multidose(t_md,
                            dose     = ___,        # total dose in mg
                            ka       = params5[["ka"]],
                            CL       = ___,        # total CL in L/h
                            V        = ___,        # total V in L
                            interval = tau,
                            n_doses  = ceiling(72 / tau) + 1),
    Interval = paste0("q", tau, "h")
  )
})

sim_df3 <- do.call(rbind, sim_list)

ggplot(sim_df3, aes(x = Time, y = conc, color = Interval)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = 5,  linetype = "dashed", color = "darkgreen") +
  geom_hline(yintercept = 15, linetype = "dashed", color = "red") +
  annotate("text", x = 70, y = 5.6,  label = "MEC (5 mg/L)",  hjust = 1, size = 3.5) +
  annotate("text", x = 70, y = 15.6, label = "MTC (15 mg/L)", hjust = 1, size = 3.5) +
  labs(
    title = "Multiple-Dose Simulation: Subject 5",
    x = "Time (h)", y = "Concentration (mg/L)", color = "Interval"
  ) +
  theme_minimal()

In [ ]:
# Check steady-state trough and peak concentrations for each regimen
# "Steady state" is approximately reached after 4-5 half-lives
# Half-life of Subject 5:
cat(sprintf("Subject 5 half-life: %.1f h\n", t_half5))
cat(sprintf("Steady state expected after: %.0f h (4-5 half-lives)\n", 4 * t_half5))

# Extract concentrations at steady state (last dosing interval, t > 48h)
ss_data <- sim_df3 %>%
  filter(Time > 48) %>%
  group_by(Interval) %>%
  summarise(
    peak  = max(conc),
    trough = min(conc),
    .groups = "drop"
  )

print(ss_data)

**Question 3a**: Which dosing interval keeps the steady-state trough above the MEC (5 mg/L) AND the peak below the MTC (15 mg/L)? Quote the trough and peak values from your table.

*Your answer:*
[Write here]

---

**Question 3b**: Subject 1 had a longer half-life than Subject 5. If you ran the same simulation for Subject 1, would you expect steady state to be reached sooner or later? Would the peak concentrations at steady state be higher or lower (for the same dose)?

*Your answer:*
[Write here]

---

**Question 3c** *(stretch)*: The simulation uses Subject 5's *individual* PK parameters. In a real clinic, you wouldn't have these — you would only know the drug's *population average* parameters. Would you design the dosing regimen more or less conservatively in that case? Why?

*Your answer:*
[Write here]

## Exercise 4: PD Modeling — Potency, Efficacy, and the Emax Model

In this exercise you will work with synthetic PD data. We will generate effect measurements for Subject 4 using a known Emax model (with noise), then fit the model and recover the parameters.

After fitting, you will compare two hypothetical drugs and reason about which is more useful clinically.

In [ ]:
set.seed(7)   # different seed from the lecture — gives different noise

sub4 <- subset(Theoph, Subject == 4)

# True PD parameters for Subject 4 (we pretend these are unknown)
Emax_true <- 80    # note: NOT 100 — this drug is a partial agonist
EC50_true <- 3     # mg/L

# Generate noisy effect data
sub4$effect <- emax_model(sub4$conc, Emax_true, EC50_true) +
               rnorm(nrow(sub4), mean = 0, sd = 5)
sub4$effect <- pmax(sub4$effect, 0)

# Plot the raw concentration-effect data
ggplot(sub4, aes(x = conc, y = effect)) +
  geom_point(size = 3, color = "steelblue") +
  labs(
    title = "Subject 4: Concentration-Effect Data",
    x = "Concentration (mg/L)", y = "Effect (%)"
  ) +
  theme_minimal()

In [ ]:
# Fit the Emax model to Subject 4's data
# Hint: use nls() just like for PK, but the model formula is:
#   effect ~ Emax * conc / (EC50 + conc)
# Starting values: Emax around 70, EC50 around 5

fit_pd4 <- nls(
  effect ~ ___ * conc / (___ + conc),   # YOUR CODE: fill in parameter names
  data  = sub4,
  start = list(Emax = ___, EC50 = ___)  # YOUR CODE: fill in starting values
)

summary(fit_pd4)
pd_params4 <- coef(fit_pd4)
cat(sprintf("\nFitted Emax = %.1f  (true = %d)\n", pd_params4["Emax"], Emax_true))
cat(sprintf("Fitted EC50 = %.2f mg/L  (true = %.1f mg/L)\n", pd_params4["EC50"], EC50_true))

In [ ]:
# Plot the fitted Emax curve over the data
C_fine <- seq(0, max(sub4$conc) * 1.2, by = 0.05)
pd_pred <- data.frame(
  conc   = C_fine,
  effect = emax_model(C_fine, pd_params4[["Emax"]], pd_params4[["EC50"]])
)

ggplot() +
  geom_point(data = sub4,    aes(x = conc, y = effect), size = 3, color = "steelblue") +
  geom_line( data = pd_pred, aes(x = conc, y = effect), color = "firebrick", linewidth = 1) +
  geom_hline(yintercept = pd_params4[["Emax"]], linetype = "dotted", color = "gray50") +
  annotate("text", x = 0.5, y = pd_params4[["Emax"]] + 2,
           label = paste0("Emax = ", round(pd_params4[["Emax"]], 1), "%"),
           hjust = 0, size = 3.5) +
  labs(
    title = "Subject 4: Emax Model Fit",
    x = "Concentration (mg/L)", y = "Effect (%)"
  ) +
  theme_minimal()

In [ ]:
# Compare two drugs on the same plot:
# Drug X: Emax = 80,  EC50 = 3  mg/L  (your fitted Drug — partial agonist)
# Drug Y: Emax = 100, EC50 = 10 mg/L  (full agonist, less potent)

C_seq <- seq(0, 20, by = 0.1)

compare_df <- data.frame(
  C     = rep(C_seq, 2),
  Drug  = rep(c("Drug X (Emax=80, EC50=3)", "Drug Y (Emax=100, EC50=10)"), each = length(C_seq)),
  Emax  = rep(c(80, 100), each = length(C_seq)),
  EC50  = rep(c(3, 10),   each = length(C_seq))
) %>%
  mutate(Effect = emax_model(C, Emax, EC50))

ggplot(compare_df, aes(x = C, y = Effect, color = Drug)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = 50, linetype = "dashed", alpha = 0.5) +
  labs(
    title = "Drug X vs Drug Y: Potency and Efficacy",
    x = "Concentration (mg/L)", y = "Effect (%)", color = NULL
  ) +
  theme_minimal() +
  theme(legend.position = "bottom")

**Question 4a**: Your fitted Emax for Subject 4 is around 80%, not 100%. What does this mean biologically? What is the term for a drug that cannot achieve full receptor activation even at very high concentrations?

*Your answer:*
[Write here]

---

**Question 4b**: Look at the comparison plot. At a concentration of 5 mg/L, which drug produces a higher effect? At 15 mg/L, which drug produces a higher effect? What changed between these two scenarios?

*Your answer:*
[Write here]

---

**Question 4c**: If you wanted to achieve 60% effect with Drug X and with Drug Y, what concentration would each require? You can read this from the plot or calculate it algebraically from $E = E_{max} C / (EC_{50} + C)$ solved for $C$:
$$C = \frac{E \cdot EC_{50}}{E_{max} - E}$$

Show your work (R code or algebra).

In [ ]:
# Calculate the concentration needed for 60% effect for each drug
target_effect <- 60   # %

# Formula: C = (E * EC50) / (Emax - E)
C_drugX <- (target_effect * ___) / (___ - target_effect)   # Drug X params
C_drugY <- (target_effect * ___) / (___ - target_effect)   # Drug Y params

cat(sprintf("Concentration for %d%% effect:\n", target_effect))
cat(sprintf("  Drug X: %.2f mg/L\n", C_drugX))
cat(sprintf("  Drug Y: %.2f mg/L\n", C_drugY))

**Question 4d**: Drug X cannot achieve 100% effect even at very high doses. In a clinical context, is this always a problem? Can you think of a situation where a partial agonist (Emax < 100%) might actually be *preferred* over a full agonist?

*Your answer:*
[Write here]

## Exercise 5 (Bonus): Population Variability and Its Clinical Implications

In the lecture, we fit a nonlinear mixed-effects (NLME) model to all 12 subjects simultaneously. In this exercise you will:
1. Fit the oral model to **each of the 12 subjects individually** using a loop
2. Build a table of individual parameter estimates
3. Visualize the variability in CL and V across the population
4. Reason about what this variability means for dosing

This gives you a simpler (but less statistically principled) view of the same thing NLME does automatically.

In [ ]:
# Fit the model to each subject in a loop
# We'll store results in a list and then combine into a data frame

subjects <- unique(Theoph$Subject)
results  <- list()

for (subj in subjects) {
  # Extract this subject's data
  dat <- subset(Theoph, Subject == subj)
  
  # Try to fit (use tryCatch in case fitting fails for some subjects)
  fit <- tryCatch(
    nls(
      conc ~ one_comp_oral(Time, Dose, ka, CL, V),
      data  = dat,
      start = c(ka = 1.5, CL = ___, V = ___)   # YOUR CODE: starting values
    ),
    error = function(e) NULL
  )
  
  if (!is.null(fit)) {
    p <- coef(fit)
    results[[length(results) + 1]] <- data.frame(
      Subject = subj,
      Wt      = dat$Wt[1],
      ka      = p[["ka"]],
      CL      = p[["CL"]],
      V       = p[["V"]],
      t_half  = log(2) / (p[["CL"]] / p[["V"]])
    )
  }
}

# Combine all results into one data frame
params_all <- do.call(rbind, results)
print(params_all)

In [ ]:
# Visualize the distribution of CL and V across subjects
# Use geom_point() for a strip plot (one dot per subject)
# Add a vertical line at the mean

p_CL <- ggplot(params_all, aes(x = CL, y = reorder(Subject, CL))) +
  geom_point(size = 3, color = "steelblue") +
  geom_vline(xintercept = mean(params_all$CL), linetype = "dashed", color = "firebrick") +
  labs(title = "Individual CL Estimates", x = "CL (L/kg/h)", y = "Subject") +
  theme_minimal()

p_V <- ggplot(params_all, aes(x = V, y = reorder(Subject, V))) +
  geom_point(size = 3, color = "darkorange") +
  geom_vline(xintercept = mean(params_all$V), linetype = "dashed", color = "firebrick") +
  labs(title = "Individual V Estimates", x = "V (L/kg)", y = "Subject") +
  theme_minimal()

grid.arrange(p_CL, p_V, ncol = 2)

In [ ]:
# Calculate coefficient of variation (CV%) for each parameter
# CV% = (SD / mean) * 100

cv_CL   <- (sd(params_all$CL) / mean(params_all$CL)) * 100
cv_V    <- (sd(params_all$V)  / mean(params_all$V))  * 100
cv_half <- (sd(params_all$t_half) / mean(params_all$t_half)) * 100

cat("=== Population Variability Summary ===\n")
cat(sprintf("  CL:     mean = %.4f  SD = %.4f  CV = %.0f%%\n",
            mean(params_all$CL), sd(params_all$CL), cv_CL))
cat(sprintf("  V:      mean = %.3f   SD = %.3f   CV = %.0f%%\n",
            mean(params_all$V),  sd(params_all$V),  cv_V))
cat(sprintf("  t_half: mean = %.1f h  SD = %.1f h  CV = %.0f%%\n",
            mean(params_all$t_half), sd(params_all$t_half), cv_half))

# What fraction of subjects have CL above the mean?
cat(sprintf("\n  Subjects with CL > mean: %d / %d\n",
            sum(params_all$CL > mean(params_all$CL)),
            nrow(params_all)))

In [ ]:
# Is CL correlated with body weight?
# Plot CL vs body weight (Wt) across subjects

ggplot(params_all, aes(x = Wt, y = CL)) +
  geom_point(size = 3, color = "steelblue") +
  geom_smooth(method = "lm", se = TRUE, color = "firebrick") +
  labs(
    title = "Is Clearance Correlated with Body Weight?",
    x = "Body Weight (kg)", y = "CL (L/kg/h)"
  ) +
  theme_minimal()

**Question 5a**: What is the CV% of CL across subjects? Is this high or low? As a rough guideline, CV% > 30% is considered substantial variability that can translate to meaningful differences in drug exposure at a fixed dose.

*Your answer:*
[Write here]

---

**Question 5b**: Find the subject with the highest CL and the subject with the lowest CL. If both received the same fixed dose (not weight-adjusted), how different would their AUC be? (Recall: for a linear model, AUC = Dose / CL.)

*Your answer:*
[Write here]

---

**Question 5c**: Look at the CL vs body weight plot. Is there a clear relationship? What does this mean for the value of weight-adjusted dosing (mg/kg rather than a fixed dose)?

*Your answer:*
[Write here]

---

**Question 5d** *(big picture)*: Based on everything you have done in these exercises, write 2–3 sentences explaining why a one-size-fits-all fixed dose is rarely optimal for drugs with narrow therapeutic windows like theophylline. What information would you want to measure or know about a patient before choosing their dose?

*Your answer:*
[Write here]

---
## Wrapping Up

Bring your completed notebook to next week's session. We will work through the answers together and discuss:
- Where your fits and simulations agreed or disagreed with expectations
- The clinical implications of the variability you found
- How population PK/PD models are used in real drug development

**Things to think about before next class:**
- Subject 5 cleared theophylline much faster than Subject 1. If both received the same dose, what would their concentration-time profiles look like overlaid?
- The Emax model is saturating. What does this imply for the dose-response relationship at high doses — and for drug safety?
- We modeled PK and PD separately. In practice, a drug's effect depends on the concentration *at the target site* (e.g., lung tissue for theophylline), not necessarily in plasma. What additional information would you need to model that?